# Attention Core In Triton

> **GPU required:** this notebook is meant to run on an NVIDIA GPU runtime with CUDA and Triton installed.

We are going to learn Triton from the bottom up. The simple idea is:

```text
A GPU kernel is a small program that runs many times in parallel on the GPU.
```

For attention, the GPU has to do a few repeated jobs very quickly:

```text
read vectors -> multiply -> reduce -> softmax -> weighted sum -> write output
```

So we will build those jobs one by one:

```text
copy -> elementwise -> sum/max/norm -> dot -> GEMV -> GEMM -> softmax -> attention
```

By the end, single-query decode attention should not look like magic. It should
look like a composition of kernels we already understand.


## Understanding GPU Architecture Before Triton

Before CUDA or Triton, we need a few GPU words. We will use a very small example:

```text
x = [10, 20, 30, 40, 50, 60, 70, 80]
y = x + 1
```

A CPU could loop through this one element at a time:

```text
for i in range(8):
    y[i] = x[i] + 1
```

A GPU tries to do many of those `i` positions at the same time.

### Kernel

A **kernel** is the function we ask the GPU to run in parallel.

```text
kernel: add_one
job:    y[i] = x[i] + 1 for many i values
```

The CPU launches the kernel. The GPU runs many copies of that work.

### Thread

A **thread** is one tiny worker on the GPU. In CUDA thinking, one thread often
handles one element or a small piece of work.

```text
thread 0 handles i = 0
thread 1 handles i = 1
thread 2 handles i = 2
...
```

So for our example:

```text
thread 0: y[0] = x[0] + 1
thread 1: y[1] = x[1] + 1
thread 2: y[2] = x[2] + 1
```

### Block

A **block** is a group of threads launched together.

```text
block 0: threads for i = 0, 1, 2, 3
block 1: threads for i = 4, 5, 6, 7
```

Blocks matter because the GPU schedules work in groups, not as one giant loop.

### Warp

A **warp** is a smaller group of threads that execute together on NVIDIA GPUs.
Usually a warp has 32 threads.

```text
warp = 32 threads moving together
```

For now, the main idea is enough: GPUs like doing the same kind of operation on
many values at once.

### SM

An **SM** means Streaming Multiprocessor. It is a compute unit inside the GPU.
A GPU has many SMs. Each SM runs blocks/warps of work.

```text
GPU
+--------------------------------------+
| SM 0 runs some blocks                |
| SM 1 runs some blocks                |
| SM 2 runs some blocks                |
| ...                                  |
+--------------------------------------+
```

If the GPU is the whole factory, an SM is one workshop inside the factory.

### Global Memory

**Global memory** is the GPU's big memory, where tensors live. When we create a
CUDA tensor in Torch, its data is stored in GPU global memory.

```text
x = torch.randn(..., device="cuda")
```

Triton kernels read from and write to this memory with:

```text
tl.load(...)
tl.store(...)
```

### CUDA vs Triton

CUDA exposes the thread/block view directly. You write code like:

```text
which block am I in?
which thread am I in?
which scalar element should this thread compute?
```

Triton gives us a higher-level view. Instead of writing one scalar thread at a
time, one Triton **program** handles a block of tensor elements.

```text
CUDA thinking:
  thread 0 handles x[0]
  thread 1 handles x[1]
  thread 2 handles x[2]

Triton thinking:
  program 0 handles x[0:4]
  program 1 handles x[4:8]
```

That is why Triton code often starts like this:

```text
pid     = tl.program_id(0)
offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
```

For `BLOCK_SIZE = 4`:

```text
program 0 offsets = [0, 1, 2, 3]
program 1 offsets = [4, 5, 6, 7]
```

So the first mental model is:

```text
CUDA thread:     usually one tiny scalar worker
Triton program:  one block/tile worker that handles many elements at once
```

Triton still becomes GPU code. It just lets us write the kernel at the block/tile
level, which is a better fit for attention kernels.

## GPU Memory: Registers, SRAM, DRAM

A GPU is not only compute units. It also has a memory hierarchy. The same value
can be cheap or expensive to use depending on where it lives.

Simple picture:

```text
fastest / smallest

registers        per thread / per program values
    |
SRAM-like memory  shared memory + L1 cache near an SM
    |
L2 cache          shared across SMs
    |
DRAM / HBM        big GPU global memory where tensors live

slowest / largest
```

### DRAM / HBM / Global Memory

When we create a CUDA tensor in Torch:

```python
x = torch.randn(1024, device="cuda")
```

its elements live in GPU global memory. This is the large memory on the GPU,
often called DRAM or HBM.

```text
large capacity
high bandwidth
much slower than registers/SRAM
```

In Triton, this is what pointers refer to:

```text
x_ptr -> address in global memory
```

And this is how we read/write it:

```text
tl.load(x_ptr + offsets)      # global memory -> registers
tl.store(y_ptr + offsets, v)  # registers -> global memory
```

### Registers

Registers are tiny, very fast storage close to the execution lanes. When a
Triton program does this:

```python
x = tl.load(x_ptr + offsets)
y = x * 2.0 + 1.0
```

`x` and `y` are not Python lists. They are block vectors held in GPU registers
while the program is running.

```text
registers are fast
registers are limited
using too many registers can reduce how many programs/warps fit on an SM
```

### SRAM-Like Memory: Shared Memory And L1 Cache

GPUs also have small fast memory near each SM. CUDA programmers often explicitly
use **shared memory**. It is SRAM-like storage that threads in a block can share.
There is also L1 cache near the SM.

For this first Triton notebook, we mostly let Triton/compiler/hardware manage
this level. Later, when we study optimized attention and FlashAttention-style
kernels, this memory level becomes very important because we want to reuse tiles
of `Q`, `K`, and `V` instead of rereading everything from DRAM.

### Why This Matters For Attention

Attention can be memory hungry. During decode, a query may need to read many key
and value vectors from the KV cache:

```text
q current token: small
K cache: many previous key vectors
V cache: many previous value vectors
```

The expensive part is often moving K/V data from global memory, not only doing
the multiply-add math.

A useful mental model:

```text
Torch CUDA tensor      lives in global memory / DRAM
Triton tl.load         brings a block into registers
Triton math            happens mostly on register values
Triton tl.store        writes results back to global memory
shared memory / cache  helps reuse nearby data in optimized kernels
```

In this notebook we will mostly write direct global-memory kernels. That is the
right first step. Later we can ask: how do optimized attention kernels reduce
DRAM reads and reuse data better?


## What Attention Needs From The GPU

Attention looks complicated, but the core math is built from a few primitives:

```text
scores = q @ K.T
  needs dot product or GEMV
    needs load q
    needs load K rows
    needs multiply
    needs sum

weights = softmax(scores)
  needs max
  needs exp
  needs sum
  needs divide

out = weights @ V
  needs load V rows
  needs multiply by weights
  needs sum
  needs store output
```

That is why this notebook does not start with a full attention kernel. We first
learn the small operations that attention depends on.


In [9]:
!nvidia-smi

Sat Jun  6 10:43:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             28W /   70W |     123MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Setup

This notebook expects CUDA and Triton. If either one is missing, stop and switch
to a GPU runtime before continuing.

`nvidia-smi` tells us the GPU, driver, and memory state. The setup cell imports
Torch and Triton, then prints the exact runtime we are using.


In [10]:
import math
import time

import torch
import triton
import triton.language as tl

assert torch.cuda.is_available(), "This notebook requires an NVIDIA GPU with CUDA."


def print_table(headers, rows):
    widths = [len(str(header)) for header in headers]
    for row in rows:
        widths = [max(width, len(str(value))) for width, value in zip(widths, row, strict=True)]
    template = "  ".join(f"{{:<{width}}}" for width in widths)
    print(template.format(*headers))
    print(template.format(*["-" * width for width in widths]))
    for row in rows:
        print(template.format(*[str(value) for value in row]))


def next_power_of_2(value):
    return 1 << (value - 1).bit_length()


def max_abs_error(actual, expected):
    return (actual - expected).abs().max().item()


print_table(
    ["item", "value"],
    [
        ["torch", torch.__version__],
        ["triton", triton.__version__],
        ["cuda device", torch.cuda.get_device_name(0)],
    ],
)

torch.manual_seed(0)


item         value       
-----------  ------------
torch        2.11.0+cu128
triton       3.6.0       
cuda device  Tesla T4    


## How To Read The Code Cells

Every Triton example has two parts:

```text
@triton.jit kernel function
  this is the GPU program

Python wrapper function
  this allocates output tensors and launches the kernel
```

The launch syntax is:

```python
kernel[grid](arguments, constexpr_arguments)
```

Read it as:

```text
launch this kernel over this many programs
```

For example:

```python
grid = (triton.cdiv(n, BLOCK_SIZE),)
copy_kernel[grid](x, y, n, BLOCK_SIZE=BLOCK_SIZE)
```

means:

```text
launch enough programs so every element from 0 to n-1 is covered
```

`tl.constexpr` means the value is known at compile time. Triton uses that to
specialize and optimize the kernel for shapes like `BLOCK_SIZE`, `D`, or tile
sizes.

## Triton Mental Model

A Triton kernel launches many **programs**. Each program owns one block of work.
Inside a program, `tl.arange` creates vector offsets for that block.

For a 1D tensor, the pattern is:

```text
pid       = tl.program_id(0)
offsets   = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
mask      = offsets < n
values    = tl.load(ptr + offsets, mask=mask)
result    = do math on values
tl.store(out_ptr + offsets, result, mask=mask)
```

A concrete example with `n = 10` and `BLOCK_SIZE = 4`:

```text
program 0 offsets: [0, 1, 2, 3]      valid: [T, T, T, T]
program 1 offsets: [4, 5, 6, 7]      valid: [T, T, T, T]
program 2 offsets: [8, 9, 10, 11]    valid: [T, T, F, F]
```

The mask prevents invalid memory reads and writes in the last partial block.
This same idea appears everywhere: copy kernels, softmax kernels, GEMM kernels,
and attention kernels.


## 1. Copy Kernel

This is the smallest useful Triton kernel:

```text
y[i] = x[i]
```

What happens:

```text
1. The grid launches enough programs to cover all elements.
2. Each program computes the offsets it owns.
3. `tl.load` reads a block from `x`.
4. `tl.store` writes that block into `y`.
5. The mask protects the last partial block.
```

Read the code with this sentence in mind: "one program copies one block."


In [11]:
@triton.jit
def copy_kernel(x_ptr, y_ptr, n: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    # This program owns one contiguous block of elements.
    pid = tl.program_id(axis=0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    # The last block may run past the end of the tensor, so we mask it.
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    tl.store(y_ptr + offsets, x, mask=mask)


def copy_triton(x, block_size=256):
    y = torch.empty_like(x)
    grid = (triton.cdiv(x.numel(), block_size),)
    copy_kernel[grid](x, y, x.numel(), BLOCK_SIZE=block_size)
    return y


x = torch.arange(1000, device="cuda", dtype=torch.float32)
y = copy_triton(x)
print("allclose", torch.allclose(y, x))
print("max_abs_error", max_abs_error(y, x))

allclose True
max_abs_error 0.0


## 2. Elementwise Kernel

Now we copy the same block pattern, but add math between load and store:

```text
y[i] = x[i] * scale + bias
```

This teaches a common GPU-kernel shape:

```text
load block -> compute block -> store block
```

Many real kernels are this idea with more expensive math. The important thing is
that `x` is not one scalar. Inside a Triton program, `x` is a vector of
`BLOCK_SIZE` values.


In [12]:
@triton.jit
def affine_kernel(
    x_ptr,
    y_ptr,
    n: tl.constexpr,
    scale: tl.constexpr,
    bias: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
):
    # This program owns one contiguous block of elements.
    pid = tl.program_id(axis=0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    # The last block may run past the end of the tensor, so we mask it.
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    y = x * scale + bias
    tl.store(y_ptr + offsets, y, mask=mask)


def affine_triton(x, scale, bias, block_size=256):
    y = torch.empty_like(x)
    grid = (triton.cdiv(x.numel(), block_size),)
    affine_kernel[grid](x, y, x.numel(), scale, bias, BLOCK_SIZE=block_size)
    return y


x = torch.randn(1024, device="cuda")
expected = x * 2.0 + 0.5
actual = affine_triton(x, scale=2.0, bias=0.5)
print("allclose", torch.allclose(actual, expected))
print("max_abs_error", max_abs_error(actual, expected))

allclose True
max_abs_error 0.0


## 3. Sum Reduction

A reduction turns many values into one value:

```text
[x0, x1, x2, ..., xN] -> x0 + x1 + x2 + ... + xN
```

Attention needs reductions because a dot product is a reduction:

```text
dot(q, k) = sum(q[d] * k[d])
```

In this first version, one Triton program reduces one full vector. That is simple
and good for learning. For very large vectors, production kernels split the work
across multiple programs and reduce partial sums later.


In [13]:
@triton.jit
def sum_kernel(x_ptr, out_ptr, n: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    offsets = tl.arange(0, BLOCK_SIZE)
    # The last block may run past the end of the tensor, so we mask it.
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    total = tl.sum(x, axis=0)
    tl.store(out_ptr, total)


def sum_triton(x):
    out = torch.empty((), device=x.device, dtype=x.dtype)
    block_size = next_power_of_2(x.numel())
    sum_kernel[(1,)](x, out, x.numel(), BLOCK_SIZE=block_size)
    return out


x = torch.randn(1024, device="cuda")
actual = sum_triton(x)
expected = x.sum()
print("actual", actual.item())
print("expected", expected.item())
print("max_abs_error", max_abs_error(actual, expected))

actual 36.29324722290039
expected 36.293243408203125
max_abs_error 3.814697265625e-06


## 4. Max Reduction

`max` is another reduction:

```text
[x0, x1, x2, ..., xN] -> largest value
```

Softmax needs this for numerical stability:

```text
softmax(x) = exp(x - max(x)) / sum(exp(x - max(x)))
```

Without subtracting `max(x)`, large positive scores can make `exp(score)` overflow.
So this kernel is directly preparing us for attention, not just doing a random
math exercise.


In [14]:
@triton.jit
def max_kernel(x_ptr, out_ptr, n: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    offsets = tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask, other=-float("inf"))
    value = tl.max(x, axis=0)
    tl.store(out_ptr, value)


def max_triton(x):
    out = torch.empty((), device=x.device, dtype=x.dtype)
    block_size = next_power_of_2(x.numel())
    max_kernel[(1,)](x, out, x.numel(), BLOCK_SIZE=block_size)
    return out


x = torch.randn(1024, device="cuda")
actual = max_triton(x)
expected = x.max()
print("actual", actual.item())
print("expected", expected.item())
print("max_abs_error", max_abs_error(actual, expected))

actual 3.4899697303771973
expected 3.4899697303771973
max_abs_error 0.0


## 5. Norm

Norm combines elementwise math and reduction:

```text
norm(x) = sqrt(sum(x * x))
```

Why include it here?

```text
1. It reuses the same load pattern.
2. It squares each element independently.
3. It reduces the squared values with `tl.sum`.
4. It applies one final scalar operation, `tl.sqrt`.
```

This pattern will be useful later for understanding normalization layers around
attention, even though norm itself is not attention.


In [15]:
@triton.jit
def l2_norm_kernel(x_ptr, out_ptr, n: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    offsets = tl.arange(0, BLOCK_SIZE)
    # The last block may run past the end of the tensor, so we mask it.
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    squared_sum = tl.sum(x * x, axis=0)
    norm = tl.sqrt(squared_sum)
    tl.store(out_ptr, norm)


def l2_norm_triton(x):
    out = torch.empty((), device=x.device, dtype=x.dtype)
    block_size = next_power_of_2(x.numel())
    l2_norm_kernel[(1,)](x, out, x.numel(), BLOCK_SIZE=block_size)
    return out


x = torch.randn(1024, device="cuda")
actual = l2_norm_triton(x)
expected = torch.linalg.vector_norm(x)
print("actual", actual.item())
print("expected", expected.item())
print("max_abs_error", max_abs_error(actual, expected))

actual 32.79519271850586
expected 32.795188903808594
max_abs_error 3.814697265625e-06


## 6. Dot Product

One attention score is one dot product:

```text
score_t = q dot k_t
        = sum(q[d] * k_t[d])
```

Shapes:

```text
q   = [D]
k_t = [D]
out = scalar
```

What the kernel does:

```text
load q block
load k block
multiply elementwise
sum over D
store one scalar score
```

This is the first kernel that looks like attention math.


In [16]:
@triton.jit
def dot_kernel(q_ptr, k_ptr, out_ptr, d: tl.constexpr, BLOCK_D: tl.constexpr):
    offsets = tl.arange(0, BLOCK_D)
    mask = offsets < d
    q = tl.load(q_ptr + offsets, mask=mask, other=0.0)
    k = tl.load(k_ptr + offsets, mask=mask, other=0.0)
    dot = tl.sum(q * k, axis=0)
    tl.store(out_ptr, dot)


def dot_triton(q, k):
    out = torch.empty((), device=q.device, dtype=q.dtype)
    block_d = next_power_of_2(q.numel())
    dot_kernel[(1,)](q, k, out, q.numel(), BLOCK_D=block_d)
    return out


d = 64
q = torch.randn(d, device="cuda")
k = torch.randn(d, device="cuda")
actual = dot_triton(q, k)
expected = torch.dot(q, k)
print("actual", actual.item())
print("expected", expected.item())
print("max_abs_error", max_abs_error(actual, expected))

actual 3.10445499420166
expected 3.1044540405273438
max_abs_error 9.5367431640625e-07


## 7. GEMV For Attention Scores

GEMV means matrix-vector multiplication:

```text
scores = K @ q
```

Shapes:

```text
K      = [T, D]   # one key vector per cached token
q      = [D]      # current query vector
scores = [T]      # one score per cached token
```

This is decode attention scoring. We have one current query, so this is not a
full matrix-matrix multiply yet.

The kernel mapping is simple:

```text
program 0 computes score for token 0
program 1 computes score for token 1
program 2 computes score for token 2
...
```

Each program loads the same `q`, loads one row from `K`, computes a dot product,
and stores one score.


In [17]:
@triton.jit
def gemv_scores_kernel(
    q_ptr,
    k_ptr,
    scores_ptr,
    t: tl.constexpr,
    d: tl.constexpr,
    BLOCK_D: tl.constexpr,
):
    row = tl.program_id(axis=0)
    offsets_d = tl.arange(0, BLOCK_D)
    mask_d = offsets_d < d
    q = tl.load(q_ptr + offsets_d, mask=mask_d, other=0.0)
    k = tl.load(k_ptr + row * d + offsets_d, mask=mask_d, other=0.0)
    score = tl.sum(q * k, axis=0)
    tl.store(scores_ptr + row, score, mask=row < t)


def gemv_scores_triton(q, k):
    t, d = k.shape
    scores = torch.empty((t,), device=q.device, dtype=q.dtype)
    block_d = next_power_of_2(d)
    gemv_scores_kernel[(t,)](q, k, scores, t, d, BLOCK_D=block_d)
    return scores


t, d = 128, 64
q = torch.randn(d, device="cuda")
k = torch.randn(t, d, device="cuda")
actual = gemv_scores_triton(q, k)
expected = k @ q
print("shape", list(actual.shape))
print("allclose", torch.allclose(actual, expected, atol=1e-4, rtol=1e-4))
print("max_abs_error", max_abs_error(actual, expected))

shape [128]
allclose True
max_abs_error 1.9073486328125e-06


## 8. GEMM

GEMM means matrix-matrix multiplication:

```text
C = A @ B
```

Shapes:

```text
A = [M, K]
B = [K, N]
C = [M, N]
```

This matters for attention prefill, where many query positions compare against
many key positions.

The tiled GEMM idea:

```text
one Triton program owns one tile of C, for example C[0:16, 0:16]
that tile needs a block from A and a block from B
we loop over K in chunks
we accumulate partial products into `acc`
finally we store the C tile
```

Simple picture:

```text
        B [K, N]
      +---------+
      |         |
A     |         |
[M,K] v         v
+---+  +----------------+
|   |  | C tile         |
|   |  | computed by    |
|   |  | one program    |
+---+  +----------------+
```

This kernel is educational, not a replacement for highly optimized library GEMM.


In [18]:
@triton.jit
def matmul_kernel(
    a_ptr,
    b_ptr,
    c_ptr,
    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    offsets_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offsets_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offsets_k = tl.arange(0, BLOCK_K)

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k_start in range(0, K, BLOCK_K):
        k_idx = k_start + offsets_k
        a = tl.load(
            a_ptr + offsets_m[:, None] * K + k_idx[None, :],
            mask=(offsets_m[:, None] < M) & (k_idx[None, :] < K),
            other=0.0,
        )
        b = tl.load(
            b_ptr + k_idx[:, None] * N + offsets_n[None, :],
            mask=(k_idx[:, None] < K) & (offsets_n[None, :] < N),
            other=0.0,
        )
        acc += tl.dot(a, b)

    tl.store(
        c_ptr + offsets_m[:, None] * N + offsets_n[None, :],
        acc,
        mask=(offsets_m[:, None] < M) & (offsets_n[None, :] < N),
    )


def matmul_triton(a, b, block_m=16, block_n=16, block_k=32):
    M, K = a.shape
    K_b, N = b.shape
    assert K == K_b
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = (triton.cdiv(M, block_m), triton.cdiv(N, block_n))
    matmul_kernel[grid](
        a,
        b,
        c,
        M,
        N,
        K,
        BLOCK_M=block_m,
        BLOCK_N=block_n,
        BLOCK_K=block_k,
    )
    return c


M, K, N = 32, 64, 48
a = torch.randn(M, K, device="cuda")
b = torch.randn(K, N, device="cuda")
actual = matmul_triton(a, b)
expected = a @ b
print("shape", list(actual.shape))
print("allclose", torch.allclose(actual, expected, atol=1e-3, rtol=1e-3))
print("max_abs_error", max_abs_error(actual, expected))

shape [32, 48]
allclose True
max_abs_error 7.62939453125e-06


## 9. Softmax

Softmax turns raw scores into weights:

```text
weights[i] = exp(score[i]) / sum(exp(score))
```

The stable version is:

```text
shifted = scores - max(scores)
weights = exp(shifted) / sum(exp(shifted))
```

The kernel is now easy to read because we already learned the pieces:

```text
load scores
max reduction
subtract max
exp each value
sum reduction
divide each value by the sum
store weights
```


In [19]:
@triton.jit
def softmax_kernel(x_ptr, y_ptr, n: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    offsets = tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask, other=-float("inf"))
    x = x - tl.max(x, axis=0)
    numerator = tl.exp(x)
    denominator = tl.sum(numerator, axis=0)
    y = numerator / denominator
    tl.store(y_ptr + offsets, y, mask=mask)


def softmax_triton(x):
    y = torch.empty_like(x)
    block_size = next_power_of_2(x.numel())
    softmax_kernel[(1,)](x, y, x.numel(), BLOCK_SIZE=block_size)
    return y


x = torch.randn(128, device="cuda") * 4
actual = softmax_triton(x)
expected = torch.softmax(x, dim=0)
print("sum", actual.sum().item())
print("allclose", torch.allclose(actual, expected, atol=1e-5, rtol=1e-5))
print("max_abs_error", max_abs_error(actual, expected))

sum 1.0
allclose True
max_abs_error 3.725290298461914e-09


## 10. Masked Softmax

Masked softmax is softmax with some positions disabled.

In attention, a mask can mean:

```text
padding token        -> do not attend
future causal token  -> do not attend
outside local window -> do not attend
not selected sparse token -> do not attend
```

The implementation idea:

```text
valid score   -> keep real score
invalid score -> pretend score is -inf
softmax       -> invalid positions become weight 0
```

This is why masks are not only for out-of-bounds memory. Masks also express the
attention pattern.


In [20]:
@triton.jit
def masked_softmax_kernel(
    x_ptr,
    y_ptr,
    n: tl.constexpr,
    valid_len: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
):
    offsets = tl.arange(0, BLOCK_SIZE)
    in_bounds = offsets < n
    valid = offsets < valid_len
    x = tl.load(x_ptr + offsets, mask=in_bounds & valid, other=-float("inf"))
    x = x - tl.max(x, axis=0)
    numerator = tl.exp(x)
    denominator = tl.sum(numerator, axis=0)
    y = numerator / denominator
    y = tl.where(valid, y, 0.0)
    tl.store(y_ptr + offsets, y, mask=in_bounds)


def masked_softmax_triton(x, valid_len):
    y = torch.empty_like(x)
    block_size = next_power_of_2(x.numel())
    masked_softmax_kernel[(1,)](
        x,
        y,
        x.numel(),
        valid_len,
        BLOCK_SIZE=block_size,
    )
    return y


x = torch.randn(128, device="cuda")
valid_len = 77
actual = masked_softmax_triton(x, valid_len)
masked = x.clone()
masked[valid_len:] = -float("inf")
expected = torch.softmax(masked, dim=0)
print("valid weight sum", actual[:valid_len].sum().item())
print("invalid weight sum", actual[valid_len:].sum().item())
print("allclose", torch.allclose(actual, expected, atol=1e-5, rtol=1e-5))
print("max_abs_error", max_abs_error(actual, expected))

valid weight sum 0.9999998807907104
invalid weight sum 0.0
allclose True
max_abs_error 7.450580596923828e-09


## 11. Single-Query Decode Attention

Now we combine the pieces into decode attention for one query:

```text
scores = K @ q / sqrt(D)
weights = softmax(scores)
out = weights @ V
```

Shapes:

```text
q   = [D]
K   = [T, D]
V   = [T, D]
out = [D]
```

What happens in plain terms:

```text
1. Load the current query vector q.
2. Load all cached key vectors K.
3. Compute one score per cached token.
4. Scale scores by sqrt(D).
5. Softmax scores into attention weights.
6. Load cached value vectors V.
7. Mix values using the weights.
8. Store the output vector.
```

This kernel intentionally keeps everything in one program for modest `T` and
`D`. That makes the code readable. A production attention kernel tiles over the
sequence, keeps a running max/sum for online softmax, and avoids materializing
large intermediate tensors.


In [21]:
@triton.jit
def single_query_attention_kernel(
    q_ptr,
    k_ptr,
    v_ptr,
    out_ptr,
    T: tl.constexpr,
    D: tl.constexpr,
    BLOCK_T: tl.constexpr,
    BLOCK_D: tl.constexpr,
):
    offsets_t = tl.arange(0, BLOCK_T)
    offsets_d = tl.arange(0, BLOCK_D)

    q = tl.load(q_ptr + offsets_d, mask=offsets_d < D, other=0.0)
    k = tl.load(
        k_ptr + offsets_t[:, None] * D + offsets_d[None, :],
        mask=(offsets_t[:, None] < T) & (offsets_d[None, :] < D),
        other=0.0,
    )
    scores = tl.sum(k * q[None, :], axis=1) / tl.sqrt(D + 0.0)
    scores = tl.where(offsets_t < T, scores, -float("inf"))

    scores = scores - tl.max(scores, axis=0)
    weights = tl.exp(scores)
    weights = weights / tl.sum(weights, axis=0)

    v = tl.load(
        v_ptr + offsets_t[:, None] * D + offsets_d[None, :],
        mask=(offsets_t[:, None] < T) & (offsets_d[None, :] < D),
        other=0.0,
    )
    out = tl.sum(weights[:, None] * v, axis=0)
    tl.store(out_ptr + offsets_d, out, mask=offsets_d < D)


def single_query_attention_triton(q, k, v):
    T, D = k.shape
    assert q.shape == (D,)
    assert v.shape == (T, D)
    out = torch.empty((D,), device=q.device, dtype=q.dtype)
    block_t = next_power_of_2(T)
    block_d = next_power_of_2(D)
    single_query_attention_kernel[(1,)](
        q,
        k,
        v,
        out,
        T,
        D,
        BLOCK_T=block_t,
        BLOCK_D=block_d,
    )
    return out


def single_query_attention_torch(q, k, v):
    scores = (k @ q) / math.sqrt(q.numel())
    weights = torch.softmax(scores, dim=0)
    return weights @ v


T, D = 128, 64
q = torch.randn(D, device="cuda")
k = torch.randn(T, D, device="cuda")
v = torch.randn(T, D, device="cuda")
actual = single_query_attention_triton(q, k, v)
expected = single_query_attention_torch(q, k, v)
print("shape", list(actual.shape))
print("allclose", torch.allclose(actual, expected, atol=1e-4, rtol=1e-4))
print("max_abs_error", max_abs_error(actual, expected))

shape [64]
allclose True
max_abs_error 5.960464477539063e-08


## 12. Benchmark The Educational Kernels

These kernels are written for understanding. They may not beat Torch. That is
fine.

At this point the benchmark is mainly checking three things:

```text
1. Did we launch real GPU kernels?
2. Are the Triton outputs close to Torch outputs?
3. How does the rough runtime compare for this small shape?
```

Later, when we understand tiling and memory movement better, we can optimize
seriously. For now, correctness and visibility matter more than winning the
benchmark.


In [22]:
def benchmark_ms(fn, *args, warmup=10, iterations=50):
    for _ in range(warmup):
        fn(*args)
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iterations):
        fn(*args)
    torch.cuda.synchronize()
    return (time.perf_counter() - start) * 1000 / iterations


T, D = 256, 64
q = torch.randn(D, device="cuda")
k = torch.randn(T, D, device="cuda")
v = torch.randn(T, D, device="cuda")

rows = [
    ["gemv scores Triton", f"{benchmark_ms(gemv_scores_triton, q, k):.4f} ms"],
    ["gemv scores Torch", f"{benchmark_ms(lambda a, b: b @ a, q, k):.4f} ms"],
    [
        "single-query attention Triton",
        f"{benchmark_ms(single_query_attention_triton, q, k, v):.4f} ms",
    ],
    [
        "single-query attention Torch",
        f"{benchmark_ms(single_query_attention_torch, q, k, v):.4f} ms",
    ],
]
print_table(["path", "time"], rows)

path                           time     
-----------------------------  ---------
gemv scores Triton             0.0289 ms
gemv scores Torch              0.0173 ms
single-query attention Triton  0.0299 ms
single-query attention Torch   0.0754 ms


## What We Learned

We started with memory movement and ended with a small decode-attention kernel.
The key idea is that attention is not one mysterious operation. It is a stack of
simpler GPU patterns:

```text
load/store
  -> elementwise math
  -> reductions
  -> dot product
  -> GEMV/GEMM
  -> softmax
  -> weighted value sum
```

Important Triton concepts:

- `@triton.jit` compiles a Python function into a GPU kernel.
- `tl.program_id` tells a program which block/tile it owns.
- `tl.arange` creates vector offsets inside that program.
- `tl.load` and `tl.store` move blocks of data.
- masks protect invalid offsets and can also express attention visibility.
- `tl.sum` and `tl.max` are block-level reductions.
- GEMV is the decode scoring shape: one query against many cached keys.
- GEMM is the prefill scoring shape: many queries against many keys.

Next we can add the head axis. Multi-head Triton attention mostly asks how to
map programs over `[B, H, T, D]` while keeping reads and writes predictable.


1. In the copy kernel, why do we need a mask even though every program has the
same `BLOCK_SIZE`?
2. In decode attention, why is `scores = K @ q` a GEMV shape instead of a GEMM
shape?
3. In softmax, what breaks if we skip `scores - max(scores)` before `exp`?

